In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import files
uploaded = files.upload()

Saving healthcare_data_cleaning_dataset.csv to healthcare_data_cleaning_dataset.csv


In [4]:
df = pd.read_csv("/content/healthcare_data_cleaning_dataset.csv")
print(df.head())

   Patient_ID   Age  Gender       City     Diagnosis  Hospital_Visits  \
0       17270  35.0    Male  Bangalore  Hypertension               13   
1       10860  21.0  Female  Hyderabad           Flu               11   
2       15390  77.0  Female  Bangalore        Asthma                2   
3       15191  79.0  Female     Mumbai        Asthma               13   
4       15734  60.0  Female      Delhi        Asthma                1   

   Treatment_Cost  Insurance_Coverage Admission_Date  
0         41010.0                   1     2023-11-30  
1         12194.0                   1     2023-02-23  
2         45086.0                   0     2023-03-14  
3         40842.0                   0     2023-08-01  
4          9873.0                   1     2023-06-20  


In [7]:
print("Check Missing Values:")
print(df.isnull().sum())

Check Missing Values:
Patient_ID              0
Age                   600
Gender                  0
City                    0
Diagnosis               0
Hospital_Visits         0
Treatment_Cost        593
Insurance_Coverage      0
Admission_Date          0
dtype: int64


In [9]:
missing_percentage = (df.isnull().sum() / len(df)) * 100
print("Calculate Missing Percentage:")
print(missing_percentage)

Calculate Missing Percentage:
Patient_ID             0.000000
Age                   11.764706
Gender                 0.000000
City                   0.000000
Diagnosis              0.000000
Hospital_Visits        0.000000
Treatment_Cost        11.627451
Insurance_Coverage     0.000000
Admission_Date         0.000000
dtype: float64


In [11]:
print(df['Age'].describe())
df['Age'] = df['Age'].fillna(df['Age'].median())
print(" ")
print("Verify Missing Values Removed:")
print(df['Age'].isnull().sum())

count    5100.000000
mean       49.644902
std        26.924961
min         0.000000
25%        28.000000
50%        50.000000
75%        71.000000
max        99.000000
Name: Age, dtype: float64
 
Verify Missing Values Removed:
0


In [12]:
df['Treatment_Cost'] = df['Treatment_Cost'].fillna(
    df['Treatment_Cost'].median()
)
print(df['Treatment_Cost'].isnull().sum())

0


In [13]:
duplicates = df.duplicated().sum()

print("Duplicate Rows:", duplicates)

Duplicate Rows: 99


In [14]:
print("Before:", df.shape)

Before: (5100, 9)


In [15]:
df = df.drop_duplicates()
print("After:", df.shape)

After: (5001, 9)


In [20]:
invalid_age = df[
    (df['Age'] < 0) |
    (df['Age'] > 100)
]

print(invalid_age)

print(invalid_age.shape)

Empty DataFrame
Columns: [Patient_ID, Age, Gender, City, Diagnosis, Hospital_Visits, Treatment_Cost, Insurance_Coverage, Admission_Date]
Index: []
(0, 9)


In [19]:
df = df[
    (df['Age'] >= 0) &
    (df['Age'] <= 100)
]
print(df)

      Patient_ID   Age  Gender       City     Diagnosis  Hospital_Visits  \
0          17270  35.0    Male  Bangalore  Hypertension               13   
1          10860  21.0  Female  Hyderabad           Flu               11   
2          15390  77.0  Female  Bangalore        Asthma                2   
3          15191  79.0  Female     Mumbai        Asthma               13   
4          15734  60.0  Female      Delhi        Asthma                1   
...          ...   ...     ...        ...           ...              ...   
4996       16135  50.0  Female      Delhi  Hypertension               13   
4997       15573  35.0  Female      Delhi      COVID-19               19   
4998       14131  60.0  Female    Chennai  Hypertension                4   
4999       19900  16.0  Female    Chennai           Flu               16   
5095       11764  50.0  Female     Mumbai      COVID-19               15   

      Treatment_Cost  Insurance_Coverage Admission_Date  
0            41010.0         

In [21]:
Q1 = df['Treatment_Cost'].quantile(0.25)
Q3 = df['Treatment_Cost'].quantile(0.75)
IQR = Q3 - Q1
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR
outliers = df[
    (df['Treatment_Cost'] < lower_limit) |
    (df['Treatment_Cost'] > upper_limit)
]
print("Number of Outliers:", outliers.shape[0])

Number of Outliers: 50


In [28]:
lower_cap = df['Treatment_Cost'].quantile(0.05)
upper_cap = df['Treatment_Cost'].quantile(0.95)

print("Lower Cap:", lower_cap)
print("Upper Cap:", upper_cap)


Lower Cap: 3238.0
Upper Cap: 47948.0


In [36]:
print("Before Winsorization")
print("")
print(df['Treatment_Cost'].describe())
print("")
df['Treatment_Cost'] = np.where(
    df['Treatment_Cost'] < lower_cap,
    lower_cap,
    df['Treatment_Cost']
)

df['Treatment_Cost'] = np.where(
    df['Treatment_Cost'] > upper_cap,
    upper_cap,
    df['Treatment_Cost']
)
print("After Winsorization")
print(df['Treatment_Cost'].describe())
print("")
print(df[['Treatment_Cost']].head(10))
print("")
print(df.head())
print("")
print("Minimum Value:", df['Treatment_Cost'].min())
print("")
print("Maximum Value:", df['Treatment_Cost'].max())

Before Winsorization

count     5001.000000
mean     25218.878624
std      13580.445490
min       3238.000000
25%      13766.000000
50%      24797.000000
75%      36542.000000
max      47948.000000
Name: Treatment_Cost, dtype: float64

After Winsorization
count     5001.000000
mean     25218.878624
std      13580.445490
min       3238.000000
25%      13766.000000
50%      24797.000000
75%      36542.000000
max      47948.000000
Name: Treatment_Cost, dtype: float64

   Treatment_Cost
0         41010.0
1         12194.0
2         45086.0
3         40842.0
4          9873.0
5         11948.0
6         31710.0
7         25910.0
8         24797.0
9          3264.0

   Patient_ID   Age  Gender       City     Diagnosis  Hospital_Visits  \
0       17270  35.0    Male  Bangalore  Hypertension               13   
1       10860  21.0  Female  Hyderabad           Flu               11   
2       15390  77.0  Female  Bangalore        Asthma                2   
3       15191  79.0  Female     Mumbai 

In [37]:
df['Log_Treatment_Cost'] = np.log(df['Treatment_Cost'])
print(df[['Treatment_Cost', 'Log_Treatment_Cost']].head())

   Treatment_Cost  Log_Treatment_Cost
0         41010.0           10.621571
1         12194.0            9.408699
2         45086.0           10.716327
3         40842.0           10.617466
4          9873.0            9.197559


In [41]:
df['Admission_Date'] = pd.to_datetime(df['Admission_Date'])
df = df.sort_values(by='Admission_Date')
df = df.fillna(method='ffill')
print(df.head())
print("")
print(df.isnull().sum())
print("")
print(df[['Admission_Date', 'Age', 'Treatment_Cost']].head(10))
print(df.info())
df = df.fillna(method='bfill')
print(df.head())


      Patient_ID   Age  Gender     City     Diagnosis  Hospital_Visits  \
2932       19591  91.0    Male  Chennai        Asthma                3   
1551       10959  17.0    Male  Chennai  Hypertension                8   
4329       17692  64.0    Male  Chennai  Hypertension               19   
866        15966  49.0  Female   Mumbai  Hypertension               10   
1650       12356  50.0  Female    Delhi      COVID-19                4   

      Treatment_Cost  Insurance_Coverage Admission_Date  Log_Treatment_Cost  
2932         32149.0                   0     2023-01-01           10.378137  
1551         12568.0                   1     2023-01-01            9.438909  
4329         36377.0                   1     2023-01-01           10.501692  
866          43720.0                   1     2023-01-01           10.685561  
1650         24797.0                   1     2023-01-01           10.118478  

Patient_ID            0
Age                   0
Gender                0
City          

/tmp/ipykernel_2039/1525767574.py:3: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='ffill')
/tmp/ipykernel_2039/1525767574.py:10: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='bfill')
